<a href="https://colab.research.google.com/github/kylashrao/DataScience-Projects/blob/main/Project_Bank_Loan_Default_Prediction_using_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

End-to-End Project: Bank Loan Default Prediction using XGBoost

We will build an end-to-end classification project to predict whether a loan applicant will default (1) or repay successfully (0).

1. Import Libraries & Generate Dataset

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Set random seed
np.random.seed(42)

# Simulating a banking loan dataset
n_samples = 1000
income = np.random.randint(30000, 150000, size=n_samples)
credit_score = np.random.randint(550, 850, size=n_samples)
debt_to_income = np.random.uniform(0.1, 0.6, size=n_samples)
loan_amount = np.random.randint(5000, 50000, size=n_samples)

# Target logic: higher debt and lower credit score increase default probability
default_prob = (
    (debt_to_income * 2.0)
    - (credit_score / 1000)
    + (loan_amount / 100000)
    + np.random.normal(0, 0.2, size=n_samples)
)
default = (default_prob > np.percentile(default_prob, 75)).astype(int)

df = pd.DataFrame(
    {
        'Income': income,
        'Credit_Score': credit_score,
        'Debt_To_Income': debt_to_income,
        'Loan_Amount': loan_amount,
        'Default': default,
    }
)

# Features and Target
X = df[['Income', 'Credit_Score', 'Debt_To_Income', 'Loan_Amount']] #Features
y = df['Default']                                                   #Target

# Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

2. Train the XGBoost Model
Tree-based models like XGBoost handle tabular data natively without requiring feature scaling, capturing complex feature thresholds efficiently.

In [ ]:
# Initialize XGBoost Classifier for credit risk evaluation
bank_model = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42
)

# Train the model
bank_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

3. Model Evaluation & Performance Metrics

In [ ]:
# Predictions
y_pred = bank_model.predict(X_test)

# Evaluation
print('--- Bank Credit Risk Model Performance ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

--- Bank Credit Risk Model Performance ---
Accuracy: 0.8350

Confusion Matrix:
[[143  14]
 [ 19  24]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.91      0.90       157
           1       0.63      0.56      0.59        43

    accuracy                           0.83       200
   macro avg       0.76      0.73      0.74       200
weighted avg       0.83      0.83      0.83       200



4. Inspecting Risk Drivers (Feature Importance)
Banks are heavily regulated and must explain why a loan was denied. XGBoost makes this transparent via feature importances.

In [ ]:
# Extract feature importances
importances = pd.Series(bank_model.feature_importances_, index=X.columns)
print('Key Risk Drivers for Loan Defaults:')
print(importances.sort_values(ascending=False))

Key Risk Drivers for Loan Defaults:
Debt_To_Income    0.526025
Loan_Amount       0.233906
Credit_Score      0.133551
Income            0.106518
dtype: float32
